# Skin Lesion Classification using CNN (V2 — Improved)

## Semester Project – Programming for AI

### Team Members
- bscs24043 Ibrahim Butt
- bscs24095 Waize Arif
- bscs24139 Syed Jaffar Raza Kazmi
- bscs24083 Muhammad Moiz

### Objective
Build a Convolutional Neural Network (CNN) model to classify skin lesion images.

### Changes from V1
- 4 conv blocks instead of 3 (slightly deeper, not too deep for our dataset size)
- Global Average Pooling replaces the massive 25M-param FC layer
- Better data augmentation (ColorJitter, slight Affine)
- ImageNet normalization statistics
- Learning rate scheduler (ReduceLROnPlateau)
- Kaiming weight initialization
- Early stopping and best model saving
- Per-class accuracy reporting

# Python Imports

In [ ]:
!pip install torchmetrics
!pip install torchvision

import torch
import torchvision
from torch.utils.data import DataLoader, Subset
import torchmetrics
from torchmetrics import ConfusionMatrix
import matplotlib.pyplot as plt
import numpy as np
from torch import nn
from timeit import default_timer as timer
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/skin_lesion/skin_dataset.zip" "/content/"

!unzip -q "/content/skin_dataset.zip" -d "/content/"

!ls "/content/dataset"

# Check Version and Device

In [ ]:
print(f"PyTorchColab Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# HyperParameters

In [ ]:
# --- HYPERPARAMETER DASHBOARD ---

SEED = 42                # seed for reproducibility
TRAIN_SPLIT = 0.8        # train test split value, 80% train, 20% test
BATCH_SIZE = 16          # smaller batch = more weight updates per epoch (better for small data)
LEARNING_RATE = 0.0005   # conservative LR, scheduler will reduce further
EPOCHS = 60              # more room to train, early stopping will prevent overfitting
DROPOUT_RATE = 0.3       # moderate dropout
IMAGE_SIZE = 224         # image size
PATIENCE = 12            # early stopping patience (generous for small dataset)

# Transforming Data

**V2 Changes:**
- ImageNet normalization instead of generic 0.5
- Moderate augmentation: ColorJitter, slight Affine shift
- Not too aggressive (dataset is small, model needs to learn real patterns)

In [ ]:
# DATA TRANSFORMING

# ImageNet normalization stats (better than generic 0.5)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
print("Transforms defined successfully.")

# Data Loading

In [ ]:
dataset_path = "/content/dataset"

# Set seed for reproducible splits
torch.manual_seed(SEED)

train_data_full = ImageFolder(root=dataset_path, transform=train_transforms)
test_data_full  = ImageFolder(root=dataset_path, transform=test_transforms)

# 80 20 split
total_images = len(train_data_full)
train_count = int(TRAIN_SPLIT * total_images)

# Generate a shared list of shuffled numbers

indices = torch.randperm(total_images).tolist()

# train and test sets
train_dataset = Subset(train_data_full, indices[:train_count])
test_dataset  = Subset(test_data_full, indices[train_count:])

# Loaders made
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Print dataset statistics
print(f"Classes: {train_data_full.classes}")
print(f"Total images: {total_images}")
print(f"Training images: {len(train_dataset)}")
print(f"Testing images: {len(test_dataset)}")
print("Data loaded correctly. Train and Test are safely separated.")

# The CNN Class

**V2 Architecture Changes:**
- 4 conv blocks (32 -> 64 -> 128 -> 256) — one more than V1 but not too deep
- Global Average Pooling replaces the massive 25M-param FC layer (now only ~33K classifier params)
- Kaiming weight initialization for better gradient flow
- Weight decay for L2 regularization

In [ ]:
class SkinCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block 1: 224x224x3 -> 112x112x32
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Block 2: 112x112x32 -> 56x56x64
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Block 3: 56x56x64 -> 28x28x128
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Block 4: 28x28x128 -> 14x14x256
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Global Average Pooling: 14x14x256 -> 1x1x256
        # Replaces flatten + Linear(100352, 256) from V1
        # V1 had ~25.7M params in that one layer, this has 0
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Classifier Head: 256 -> 128 -> 5
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(128, 5)
        )

        # Initialize weights using Kaiming for better training
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.gap(x)             # Global Average Pooling
        x = x.view(x.size(0), -1)   # Flatten: (B, 256)

        x = self.classifier(x)
        return x

model = SkinCNN().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning rate scheduler: reduces LR when loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=4, factor=0.5
)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"CNN built and pushed to {device}.")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


# Training Loop

**V2 Changes:**
- Tracks both train loss AND test accuracy every epoch
- Learning rate scheduler adjusts LR when loss plateaus
- Early stopping prevents overfitting
- Saves best model weights

In [ ]:
train_loss_history = []
test_acc_history = []
best_accuracy = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    # ---- TRAINING ----
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_loss_history.append(epoch_loss)

    # ---- TESTING ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    test_acc_history.append(epoch_acc)

    # Step the scheduler
    scheduler.step(epoch_loss)

    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Test Acc: {epoch_acc:.2f}% | LR: {current_lr:.6f}")

    # Save best model
    if epoch_acc > best_accuracy:
        best_accuracy = epoch_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"  -> New best accuracy! Saving model.")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs (no improvement for {PATIENCE} epochs).")
        break

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nTraining complete. Best Test Accuracy: {best_accuracy:.2f}%")

# Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, len(train_loss_history)+1), train_loss_history, 'b-', linewidth=2)
ax1.set_title('Training Loss per Epoch', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(range(1, len(test_acc_history)+1), test_acc_history, 'g-', linewidth=2)
ax2.set_title('Test Accuracy per Epoch', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Accuracy and Prediction Checking

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Per-class accuracy report
print("Per-Class Classification Report:")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=train_data_full.classes))

# confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_data_full.classes)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix: Actual vs Predicted")
plt.show()